# Lip Sync Processing with Wav2Lip

This notebook processes video files to add lip synchronization using Wav2Lip.

**IMPORTANT:** Before running, switch to GPU runtime:
- Go to Runtime > Change runtime type
- Select T4 GPU
- Click Save

**Steps:**
1. Install Wav2Lip dependencies
2. Clone Wav2Lip repo and download pre-trained model
3. Upload your video and audio files
4. Run lip sync processing
5. View and download the result

## Cell 1: Setup - Install Dependencies

In [ ]:
# Install required packages for Wav2Lip
!pip install -q librosa==0.9.2 numpy==1.23.5 opencv-python-headless mediapipe
!pip install -q batch-face tqdm

# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU available: {gpu_name}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('WARNING: No GPU detected!')
    print('Please go to Runtime > Change runtime type > T4 GPU')

print('\nDependencies installed successfully!')

## Cell 2: Clone Wav2Lip Repository and Download Model

In [ ]:
import os

# Clone Wav2Lip repository
if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    print('Wav2Lip repository cloned.')
else:
    print('Wav2Lip repository already exists.')

# Create checkpoints directory
os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)

# Download pre-trained model (wav2lip_gan.pth for better quality)
model_path = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
if not os.path.exists(model_path):
    print('Downloading Wav2Lip GAN model...')
    !gdown 1PNMxQMngaoSB7KHBmOVEJFnEOqfMVT1y -O {model_path}
    print(f'Model downloaded: {model_path}')
else:
    print(f'Model already exists: {model_path}')

# Download face detection model
det_model_path = '/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth'
os.makedirs(os.path.dirname(det_model_path), exist_ok=True)
if not os.path.exists(det_model_path):
    print('Downloading face detection model...')
    !wget -q https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth -O {det_model_path}
    print('Face detection model downloaded.')
else:
    print('Face detection model already exists.')

print('\nAll models ready!')

## Cell 3: Upload Video and Audio Files

Upload your video (with a face) and audio file, OR paste Google Drive links below.

In [ ]:
from google.colab import files as colab_files
from pathlib import Path

# Option 1: Upload files directly
USE_UPLOAD = True  # Set to False to use Google Drive links instead

# Option 2: Google Drive links (set USE_UPLOAD = False)
GDRIVE_VIDEO_ID = ''  # Google Drive file ID for video
GDRIVE_AUDIO_ID = ''  # Google Drive file ID for audio

input_dir = Path('/content/lipsync_input')
input_dir.mkdir(exist_ok=True)

if USE_UPLOAD:
    print('Upload your VIDEO file (mp4, avi, mov):')
    uploaded_video = colab_files.upload()
    video_filename = list(uploaded_video.keys())[0]
    video_path = input_dir / video_filename
    with open(video_path, 'wb') as f:
        f.write(uploaded_video[video_filename])
    print(f'Video uploaded: {video_path}')
    
    print('\nUpload your AUDIO file (mp3, wav):')
    uploaded_audio = colab_files.upload()
    audio_filename = list(uploaded_audio.keys())[0]
    audio_path = input_dir / audio_filename
    with open(audio_path, 'wb') as f:
        f.write(uploaded_audio[audio_filename])
    print(f'Audio uploaded: {audio_path}')
else:
    # Download from Google Drive
    if GDRIVE_VIDEO_ID:
        !gdown {GDRIVE_VIDEO_ID} -O /content/lipsync_input/input_video.mp4
        video_path = input_dir / 'input_video.mp4'
    if GDRIVE_AUDIO_ID:
        !gdown {GDRIVE_AUDIO_ID} -O /content/lipsync_input/input_audio.mp3
        audio_path = input_dir / 'input_audio.mp3'
    print('Files downloaded from Google Drive.')

print(f'\nVideo: {video_path}')
print(f'Audio: {audio_path}')
print('\nReady for lip sync processing!')

## Cell 4: Run Wav2Lip Processing

In [ ]:
import subprocess

# Output path
output_path = '/content/lipsync_output/result.mp4'
os.makedirs('/content/lipsync_output', exist_ok=True)

# Wav2Lip parameters
WAV2LIP_DIR = '/content/Wav2Lip'
CHECKPOINT = f'{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth'

print('Starting Wav2Lip processing...')
print(f'  Video: {video_path}')
print(f'  Audio: {audio_path}')
print(f'  Output: {output_path}')
print()
print('This may take several minutes depending on video length...')
print()

# Run Wav2Lip inference
cmd = [
    'python', f'{WAV2LIP_DIR}/inference.py',
    '--checkpoint_path', CHECKPOINT,
    '--face', str(video_path),
    '--audio', str(audio_path),
    '--outfile', output_path,
    '--resize_factor', '1',
    '--nosmooth',
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print('Lip sync processing COMPLETE!')
    print(f'Output saved to: {output_path}')
    output_size = os.path.getsize(output_path) / (1024 * 1024)
    print(f'File size: {output_size:.1f} MB')
else:
    print('ERROR during processing:')
    print(result.stderr[-2000:] if result.stderr else 'No error output')
    print('\nTips:')
    print('- Make sure the video has a clearly visible face')
    print('- Try a shorter video clip first')
    print('- Ensure GPU runtime is selected')

## Cell 5: Display Result Video

In [ ]:
from IPython.display import HTML
from base64 import b64encode

if os.path.exists(output_path):
    # Display video in notebook
    video_data = open(output_path, 'rb').read()
    video_b64 = b64encode(video_data).decode()
    
    display(HTML(f"""
    <h3>Lip Sync Result</h3>
    <video width="640" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """))
else:
    print('No output video found. Please run Cell 4 first.')

## Cell 6: Download Result

In [ ]:
from google.colab import files as colab_files

if os.path.exists(output_path):
    print(f'Downloading: {output_path}')
    print(f'Size: {os.path.getsize(output_path) / (1024*1024):.1f} MB')
    colab_files.download(output_path)
    print('Download started!')
else:
    print('No output file to download. Please run the processing cells first.')